In [ ]:
import re
import zipfile
import os
from google.colab import files

# Faz upload do ZIP
uploaded = files.upload()

arquivo_zip = next(iter(uploaded))

# Pasta para extração
pasta_extraida = "arquivos_extraidos"
os.makedirs(pasta_extraida, exist_ok=True)

# Extrai o ZIP
with zipfile.ZipFile(arquivo_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_extraida)

# Procura o primeiro TXT dentro do ZIP
arquivo_txt = None

for raiz, dirs, arquivos in os.walk(pasta_extraida):
    for arquivo in arquivos:
        if arquivo.lower().endswith(".txt"):
            arquivo_txt = os.path.join(raiz, arquivo)
            break

    if arquivo_txt:
        break

if not arquivo_txt:
    raise Exception("Nenhum arquivo TXT encontrado dentro do ZIP.")

print(f"TXT encontrado: {arquivo_txt}")

# Lê o conteúdo do arquivo
with open(arquivo_txt, "r", encoding="utf-8", errors="ignore") as f:
    conteudo = f.read()

# Captura todos os blocos ZPL (^XA até ^XZ)
blocos = re.findall(r"\^XA.*?\^XZ", conteudo, re.DOTALL)

# Mostra a quantidade total encontrada antes da remoção
print(f"Total de etiquetas encontradas: {len(blocos)}")

# Remove as DANFEs
etiquetas = [
    bloco
    for bloco in blocos
    if "DANFE SIMPLIFICADO" not in bloco
]

# Calcula quantas DANFEs foram removidas
removidas = len(blocos) - len(etiquetas)

print(f"DANFEs removidas: {removidas}")
print(f"Etiquetas mantidas: {len(etiquetas)}")

# Usa o nome do ZIP enviado como nome do arquivo gerado
nome_zip = os.path.splitext(os.path.basename(arquivo_zip))[0]
arquivo_saida = f"{nome_zip}_sem_danfe.txt"

# Salva o resultado
with open(arquivo_saida, "w", encoding="utf-8") as f:
    f.write("\n".join(etiquetas))

print(f"Arquivo gerado: {arquivo_saida}")

# Download automático
files.download(arquivo_saida)